# 4 · Recomendação + Paridade (numpy / FAISS / pgvector)

Demonstra o ranking offline e prova que **numpy, FAISS e pgvector** retornam o mesmo
top-k — ou seja, o FAISS é redundante e a produção pode usar só o pgvector (backend).

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import recommend, index as faiss_index

emb = np.load(recommend.EMBEDDINGS_PATH)  # caminho absoluto (independe do CWD)
print('perfis:', emb.shape)

/Users/chaves/programacao/Trainees_2026.1-Pauta/recommendation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


perfis: (94, 768)


## A query do cidadão cai no mesmo espaço dos perfis

In [2]:
q = recommend.embed_query('saúde pública, postos de saúde e atendimento médico')
print('shape:', q.shape, '| norma L2:', round(float(np.linalg.norm(q)), 4))
assert q.shape == (768,) and abs(np.linalg.norm(q) - 1) < 1e-4

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42046.57it/s]


[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


shape: (768,) | norma L2: 1.0


## Recomendações para algumas pautas

In [3]:
for texto in ['saúde pública e creches',
              'transporte público, mobilidade e ciclovias',
              'meio ambiente, arborização e coleta de lixo']:
    print(f'\n=== {texto!r} ===')
    print(recommend.recomendar(texto, k=5).to_string(index=False))


=== 'saúde pública e creches' ===


                                            nome      municipio  cluster_id    score
                                   Mikika Leitão    João Pessoa           1 0.350805
                                  Fábio Carneiro    João Pessoa           1 0.324941
Josicleide da Silva Vicente (Cleidinha de Digão)     Santa Rita           1 0.297739
                                 Josauro Pereira         Bayeux           0 0.276032
                                Waléria Assunção Campina Grande           1 0.271234

=== 'transporte público, mobilidade e ciclovias' ===
              nome      municipio  cluster_id    score
   Josauro Pereira         Bayeux           0 0.361452
    Fábio Carneiro    João Pessoa           1 0.340877
   Luís da Padaria    João Pessoa           1 0.230310
      Plínio Gomes Campina Grande           1 0.213483
Jefferson Oliveira         Bayeux           0 0.195263

=== 'meio ambiente, arborização e coleta de lixo' ===


              nome      municipio  cluster_id    score
   Luís da Padaria    João Pessoa           1 0.318605
Jefferson Oliveira         Bayeux           0 0.309433
   Josauro Pereira         Bayeux           0 0.291704
            França         Bayeux           0 0.256260
      Plínio Gomes Campina Grande           1 0.207955


## Paridade numpy × FAISS

In [4]:
idx = faiss_index.construir_index()  # default absoluto
scores_np = emb @ q
top_np = np.argsort(-scores_np)[:5].tolist()
_, top_faiss = faiss_index.buscar(idx, q, k=5)
top_faiss = [int(i) for i in top_faiss]
print('top-5 numpy :', top_np)
print('top-5 FAISS :', top_faiss)
print('IGUAIS?', top_np == top_faiss)
assert top_np == top_faiss

[faiss] Índice construído: ntotal=94, d=768
top-5 numpy : [90, 57, 45, 40, 38]
top-5 FAISS : [90, 57, 45, 40, 38]
IGUAIS? True


## E o pgvector?

Em produção o mesmo top-k sai do Postgres: `Politico.embedding.cosine_distance(vetor)`
ordenado (índice HNSW), via `GET /recomendacoes`. Como os três usam **cosseno sobre os
mesmos vetores 768d unitários**, o ranking é idêntico — por isso o índice FAISS existe
**só para esta validação**, e a produção dispensa-o.

## Sanidade semântica: embeddar 1 ementa real recupera o próprio perfil?

In [5]:
df = pd.read_csv(recommend.MODELS_DIR.parent / 'data' / 'df_nlp_filtrado.csv').rename(columns={'vereador': 'nome'})
meta = pd.read_csv(recommend.META_PATH)
rng = np.random.RandomState(0); ranks = []
for i in rng.choice(len(meta), 8, replace=False):
    nome, mun = meta.iloc[i]['nome'], meta.iloc[i]['municipio']
    ements = df[(df['nome'] == nome) & (df['municipio'] == mun)]['proposta_ementa_filtrada'].dropna()
    ements = ements[ements != 'sem_proposta_ementa']
    if len(ements) == 0:
        continue
    qv = recommend.embed_query(max(ements, key=len))
    rank = int(np.argsort(-(emb @ qv)).tolist().index(i)) + 1
    ranks.append(rank)
print('rank do próprio perfil (de 94):', sorted(ranks))
print('mediana:', int(np.median(ranks)))

rank do próprio perfil (de 94): [1, 2, 2, 5, 6, 20, 30, 42]
mediana: 5
